In [1]:
import geopandas as gpd
import pygris
import time
import pandas as pd

osm_landuse = "../../OpenStreetMap/gis_osm_landuse_a_free_1.shp"
output = "urban_farms_with_addresses.csv"

target_counties = ["Dallas", "Harris", "Tarrant", "Travis", "Bexar"]
counties = pygris.counties(state="TX", year=2025)
counties = counties[counties["NAME"].isin(target_counties)].copy()

landuse = gpd.read_file(osm_landuse)

farms = landuse[
    (landuse["fclass"] == "allotments") |
    (
        (landuse["fclass"] == "farmland") &
        (landuse["name"].notna()) &
        (landuse["name"] != "")
    )
].copy()

if farms.crs != counties.crs:
    counties = counties.to_crs(farms.crs)

farms = gpd.sjoin(
    farms,
    counties[["NAME", "COUNTYFP", "geometry"]],
    how="inner",
    predicate="intersects"
).rename(columns={"NAME": "county"})

farms_projected = farms.to_crs("EPSG:3857")
centroids = farms_projected.geometry.centroid.to_crs("EPSG:4326")
farms["lat"] = centroids.y
farms["lon"] = centroids.x

Using FIPS code '48' for input 'TX'


In [3]:
from dotenv import load_dotenv
import os
import googlemaps
import pandas as pd
import time

load_dotenv()

api_key = os.getenv("google_api_key")
gmaps = googlemaps.Client(key=api_key)

def reverse_geocode(lat, lon):
    result = gmaps.reverse_geocode((lat, lon))
    if result:
        components = result[0]["address_components"]
        full_address = result[0]["formatted_address"]
        house_number = next((c["long_name"] for c in components if "street_number" in c["types"]), "")
        street = next((c["long_name"] for c in components if "route" in c["types"]), "")
        city = next((c["long_name"] for c in components if "locality" in c["types"]), "")
        zipcode = next((c["long_name"] for c in components if "postal_code" in c["types"]), "")
        return full_address, house_number, street, city, zipcode
    else:
        return "", "", "", "", ""

full_addresses, house_numbers, streets, cities, zipcodes = [], [], [], [], []

for i, (_, row) in enumerate(farms.iterrows()):
    print(f"  {i+1}/{len(farms)} — {row['name']}", end="\r")
    full, house, street, city, zipcode = reverse_geocode(row["lat"], row["lon"])
    full_addresses.append(full)
    house_numbers.append(house)
    streets.append(street)
    cities.append(city)
    zipcodes.append(zipcode)
    time.sleep(0.1)

farms["full_address"] = full_addresses
farms["house_number"] = house_numbers
farms["street"] = streets
farms["city"] = cities
farms["zipcode"] = zipcodes

export_cols = ["osm_id", "fclass", "name", "county", "COUNTYFP",
               "lat", "lon", "house_number", "street", "city",
               "zipcode", "full_address"]
farms[export_cols].to_csv(output, index=False)

Done!47 — Elmwood Farml Refugee Community Gardenrkpost Site
